<a href="https://colab.research.google.com/github/huwperkins08-lang/TSF_seepage_detection_ElSoldado/blob/main/REP_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ee
import geemap.core as geemap

# 1. INITIALIZATION
ee.Authenticate()
ee.Initialize(project='sxe390-el-soldado')

# 2. PARAMETERS & GEOMETRY
lat, lon = -32.65, -71.16
mine_site = ee.Geometry.Point([lon, lat])
study_area = mine_site.buffer(20000)

# 3. HELPER FUNCTIONS
def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(
           qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    return image.updateMask(mask).divide(10000).copyProperties(image, ['system:time_start'])

def resample_to_20m(image):
    bands_10m = ['B2', 'B3', 'B4', 'B8']
    bands_20m = ['B5', 'B6', 'B7', 'B8A', 'B11', 'B12']
    target_projection = image.select('B5').projection()
    downsampled = (image.select(bands_10m)
                   .reduceResolution(reducer=ee.Reducer.mean(), maxPixels=1024)
                   .reproject(crs=target_projection))
    return image.addBands(downsampled, overwrite=True).select(bands_10m + bands_20m)

def apply_terrain_mask(image):
    dem = ee.Image("NASA/NASADEM_HGT/001").select('elevation')
    slope = ee.Terrain.slope(dem)
    return image.updateMask(slope.lte(20)) # Mask slopes > 20 degrees

def apply_illumination_mask(image):
    # 1. Safely get Sun angles. If they are missing, 'zen_val' becomes None
    zen_val = image.get('MEAN_SOLAR_ZENITH_ANGLE')
    azi_val = image.get('MEAN_SOLAR_AZIMUTH_ANGLE')

    # 2. Use a conditional: If the metadata exists, do the math. If not, return the original image.
    # This prevents the 'null' multiplication error.
    return ee.Image(ee.Algorithms.If(
        zen_val, # The condition: 'Does this value exist?'
        # TRUE CASE: Do the illumination math
        (lambda: (
            image.updateMask(
                # Math moved inside here to stay safe
                (terrain_math(image, zen_val, azi_val))
            )
        ))(),
        # FALSE CASE: Just return the image without the illumination mask
        image
    ))

# This is a helper function to keep the logic clean inside the 'If' statement
def terrain_math(image, zen, azi):
    degree2rad = 3.14159 / 180.0
    z = ee.Number(zen).multiply(degree2rad)
    a = ee.Number(azi).multiply(degree2rad)

    dem = ee.Image("NASA/NASADEM_HGT/001").select('elevation')
    terrain = ee.Terrain.products(dem)
    s = terrain.select('slope').multiply(degree2rad)
    asp = terrain.select('aspect').multiply(degree2rad)

    term1 = s.cos().multiply(z.cos())
    term2 = s.sin().multiply(z.sin()).multiply(asp.subtract(a).cos())
    cos_i = term1.add(term2)
    return cos_i.gt(0.1)

def calculate_rep(image):
    b4, b5, b6, b7 = image.select('B4'), image.select('B5'), image.select('B6'), image.select('B7')
    re_reflectance = (b4.add(b7)).divide(2)
    rep = ee.Image(705).add(
        ee.Image(35).multiply(
            re_reflectance.subtract(b5).divide(b6.subtract(b5))
        )
    ).rename('REP')
    ndvi = image.normalizedDifference(['B8', 'B4'])
    return image.addBands(rep).updateMask(ndvi.gt(0.3))

# 4. SOURCE & FILTER
s2_collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                  .filterBounds(mine_site)
                  .filterDate('2017-01-01', '2025-12-31')
                  .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 40))
                  .filter(ee.Filter.notNull(['MEAN_SOLAR_ZENITH_ANGLE', 'MEAN_SOLAR_AZIMUTH_ANGLE'])))

# 5. EXECUTION PIPELINE
processed_collection = (s2_collection
                        .map(mask_s2_clouds)
                        .map(resample_to_20m)
                        .map(apply_terrain_mask)
                        .map(apply_illumination_mask) # <--- Added here
                        .map(calculate_rep))

# 6. GENERATE OUTPUT
final_median = processed_collection.median().clip(study_area)

# 7. VISUALIZATION
Map = geemap.Map(center=[lat, lon], zoom=13)
Map.addLayer(final_median, {'bands':['B4','B3','B2'], 'min':0, 'max':0.25}, 'Natural Color')

rep_params = {'bands':['REP'], 'min': 705, 'max': 720, 'palette': ['red', 'orange', 'yellow', 'green']}
Map.addLayer(final_median, rep_params, 'Red Edge Position (REP)')

Map

In [ ]:
import geopandas as gpd
import ee
import geemap # Use top-level geemap for ee_to_df

# 1. Load  QGIS GeoJSON
gdf = gpd.read_file('sites_final_300m.geojson')

# 2. Convert to Earth Engine FeatureCollection
def geodf_to_ee(gdf):
    features = []
    for _, row in gdf.iterrows():
        # Convert QGIS geometry to EE geometry using the full GeoJSON interface
        # This handles various geometry types (Polygon, MultiPolygon, etc.) robustly.
        ee_geometry = ee.Geometry(row.geometry.__geo_interface__)
        f = ee.Feature(ee_geometry, row.drop('geometry').to_dict())
        features.append(f)
    return ee.FeatureCollection(features)

study_sites = geodf_to_ee(gdf)

# 3. Extract EVERY pixel value (REP) for each site
# We use 'final_median' which is your clean, 20m, terrain-masked composite
pixel_data = final_median.sampleRegions(
    collection=study_sites,
    properties=['site_type', 'site_name'], # Use whatever columns you have in your GeoJSON
    scale=20, # Matches our resampled resolution
    geometries=True
)

# 4. Convert to DataFrame
# Updated to select 'REP' (the name used in our calculate_rep function)
df_pixels = geemap.ee_to_df(pixel_data.select(['REP', 'site_type']))

print(f"Successfully extracted {len(df_pixels)} individual pixel values!")
print(df_pixels.head())

In [ ]:
# This will show exactly how many pixels were found for each site type
print("Breakdown of pixels by Site Type:")
print(df_pixels['site_type'].value_counts())

# This will show the average REP value for Test vs Control
print("\nMean REP values:")
print(df_pixels.groupby('site_type')['REP'].mean())


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches

# 1. Prepare the Data - we ensure 'Test' is the first color (Purple)
site_names = ['Test', 'Control_1', 'Control_2', 'Control_3', 'Control_4', 'Control_5']
colors = sns.color_palette('viridis', n_colors=len(site_names))

plt.figure(figsize=(12, 7))

# 2. Draw the curves - we use the exact same order here
sns.kdeplot(data=df_pixels, x='REP', hue='site_type', hue_order=site_names,
            fill=True, common_norm=False, palette='viridis', alpha=0.3)

# 3. Add the Mean Line
test_mean = df_pixels[df_pixels['site_type'] == 'Test']['REP'].mean()
mean_line = plt.axvline(test_mean, color='red', linestyle='--', label=f'Test Mean: {test_mean:.2f}')

# 4. Create the Legend handles manually to match the order
legend_handles = []
for i, name in enumerate(site_names):
    patch = mpatches.Patch(color=colors[i], label=name, alpha=0.5)
    legend_handles.append(patch)
legend_handles.append(mean_line)

# 5. Final Plot Touches
plt.legend(handles=legend_handles, title="Study Sites", loc='upper left', frameon=True)
plt.title('Red Edge Position (REP) Distribution: El Soldado Analysis', fontsize=16)
plt.xlabel('Red Edge Position (nm)', fontsize=13)
plt.ylabel('Pixel Density', fontsize=13)

plt.show()

In [ ]:
import pandas as pd
# 1. DEFINE EXTRACTION FUNCTION
def extract_test_site_stats(image):
    # Get the date
    date = image.date().format('YYYY-MM-dd')

    # Isolate just the 'Test' site from your GeoJSON FeatureCollection
    test_geom = study_sites.filter(ee.Filter.eq('site_type', 'Test')).geometry()

    # Calculate Mean REP for this specific image over the Test site
    stats = image.select('REP').reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=test_geom,
        scale=20,
        maxPixels=1e9
    )

    # Return a Feature with the data
    return ee.Feature(None, {
        'date': date,
        'mean_rep': stats.get('REP')
    })

# 2. RUN THE HARVEST
# Note the spelling: FeatureCollection (no 'd')
temporal_fc = ee.FeatureCollection(processed_collection.map(extract_test_site_stats))

# 3. CONVERT TO DATAFRAME
df_temporal = geemap.ee_to_df(temporal_fc)

# 4. CLEANUP
df_temporal['date'] = pd.to_datetime(df_temporal['date'])
df_temporal = df_temporal.dropna().sort_values('date')

print(f"Captured {len(df_temporal)} individual satellite observations for 2023.")
df_temporal.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Set the visual style
plt.figure(figsize=(14, 7))
sns.set_theme(style="whitegrid")

# 2. Plot the raw observations
sns.scatterplot(data=df_temporal, x='date', y='mean_rep',
                color='purple', s=80, alpha=0.6, label='Individual Observation')

# 3. Add a trend line (3-point rolling average)
# This helps smooth out minor atmospheric variations
df_temporal['rolling_trend'] = df_temporal['mean_rep'].rolling(window=3, center=True).mean()
plt.plot(df_temporal['date'], df_temporal['rolling_trend'],
         color='purple', linewidth=3, label='Temporal Trend')

# 4. Add the "Health Threshold" reference
# Based on your KDE plot, the controls peaked around 720-721nm
plt.axhline(y=720.5, color='green', linestyle='--', linewidth=1.5, label='Healthy Baseline (Control Mean)')

# 5. Formatting
plt.title('2023 Red Edge Position (REP) Timeline: El Soldado Test Site', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('REP Wavelength (nm)', fontsize=12)
plt.ylim(715, 725) # Focuses the zoom on the shift
plt.legend()
plt.xticks(rotation=45)

plt.show()

In [ ]:
# 1. New Extraction Function for ALL sites
def extract_all_sites_stats(image):
    date = image.date().format('YYYY-MM-dd')

    # We loop through each site type in your FeatureCollection
    def get_site_mean(site_type):
        geom = study_sites.filter(ee.Filter.eq('site_type', site_type)).geometry()
        return image.select('REP').reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=geom,
            scale=20
        ).get('REP')

    return ee.Feature(None, {
        'date': date,
        'Test': get_site_mean('Test'),
        'Control_1': get_site_mean('Control_1'),
        'Control_2': get_site_mean('Control_2'),
        'Control_3': get_site_mean('Control_3'),
        'Control_4': get_site_mean('Control_4'),
        'Control_5': get_site_mean('Control_5')
    })

# 2. HARVEST
full_temporal_fc = ee.FeatureCollection(processed_collection.map(extract_all_sites_stats))
df_full = geemap.ee_to_df(full_temporal_fc)

# 3. CLEANUP
df_full['date'] = pd.to_datetime(df_full['date'])

# NEW: Calculate the average of controls first
control_cols = ['Control_1', 'Control_2', 'Control_3', 'Control_4', 'Control_5']
df_full['Control_Avg'] = df_full[control_cols].mean(axis=1)

# NEW: Filter out the 'Spike' (Keep only realistic REP values)
df_clean = df_full[
    (df_full['Test'] > 700) & (df_full['Test'] < 730) &
    (df_full['Control_Avg'] > 700) & (df_full['Control_Avg'] < 730)
].dropna().sort_values('date')

# 4. PLOT COMPARISON (Using the cleaned data)
plt.figure(figsize=(15, 8))

# Plot Test Site
sns.lineplot(data=df_clean, x='date', y='Test', color='purple', linewidth=2.5, label='TEST SITE (Stream/TSF)')

# Plot Control Average
sns.lineplot(data=df_clean, x='date', y='Control_Avg', color='green', linewidth=2, linestyle='--', label='CONTROLS (Regional Average)')

plt.title('2018-2025 REP Comparison: Impact vs. Regional Baseline', fontsize=16)
plt.ylabel('Red Edge Position (nm)')
plt.ylim(705, 725) # This forces the view to the relevant spectral range
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print(f"Removed outliers. Plotting {len(df_clean)} valid temporal observations.")

In [ ]:
import pandas as pd
# 1. Calculate a 6-month rolling average to smooth the 8-year noise
df_clean['Test_Smooth'] = df_clean['Test'].rolling(window=12, center=True).mean()
df_clean['Control_Smooth'] = df_clean['Control_Avg'].rolling(window=12, center=True).mean()

# 2. Plot the smoothed trends
plt.figure(figsize=(15, 8))

plt.plot(df_clean['date'], df_clean['Test_Smooth'], color='purple', linewidth=3, label='Test Site (Smoothed)')
plt.plot(df_clean['date'], df_clean['Control_Smooth'], color='green', linewidth=3, linestyle='--', label='Controls (Smoothed)')

# 3. Fill the gap - this visually highlights the 'Stress Deficit'
plt.fill_between(df_clean['date'], df_clean['Test_Smooth'], df_clean['Control_Smooth'],
                 color='gray', alpha=0.2, label='The Stress Gap')

plt.title('8-Year Vegetation Health Deficit: El Soldado (2018-2025)', fontsize=16)
plt.ylabel('Red Edge Position (nm)')
plt.ylim(714, 722)
plt.legend()
plt.show()

In [ ]:
# 1. Create a column for the 'Gap' (Control minus Test)
df_clean['Stress_Gap'] = df_clean['Control_Avg'] - df_clean['Test']

# 2. Compare the Gap in 'Dry' vs 'Wet' seasons (Approximate for Chile)
# Wet Season: June, July, August (Months 6, 7, 8)
df_clean['month'] = df_clean['date'].dt.month
df_clean['season'] = df_clean['month'].apply(lambda x: 'Wet' if x in [6, 7, 8] else 'Dry')

seasonal_gap = df_clean.groupby('season')['Stress_Gap'].mean()

print("--- Seasonal Stress Analysis ---")
print(f"Average Gap (Dry Season): {seasonal_gap['Dry']:.2f} nm")
print(f"Average Gap (Wet Season): {seasonal_gap['Wet']:.2f} nm")

# 3. Correlation Check
correlation = df_clean['Test'].corr(df_clean['Control_Avg'])
print(f"\nOverall 8-Year Correlation: {correlation:.3f}")

In [ ]:
# 1. Create a "Climatology" of the Red Edge Position
# We group by month to see the average seasonal cycle over the 8 years
monthly_climatology = df_clean.groupby('month')[['Test', 'Control_Avg']].mean().reset_index()

# 2. Plot the seasonal cycle
plt.figure(figsize=(12, 6))

plt.plot(monthly_climatology['month'], monthly_climatology['Control_Avg'],
         marker='o', color='green', linewidth=3, label='Controls (Normal Cycle)')

plt.plot(monthly_climatology['month'], monthly_climatology['Test'],
         marker='o', color='purple', linewidth=3, label='Test Site (Impacted Cycle)')

# 3. Highlight the Winter Gap (June-August)
plt.axvspan(6, 8, color='blue', alpha=0.1, label='Wet Season (Contaminant Flush)')

plt.title('The Seasonal Inversion: Average REP Cycle (2018-2025)', fontsize=15)
plt.xlabel('Month of Year', fontsize=12)
plt.ylabel('Mean Red Edge Position (nm)', fontsize=12)
plt.xticks(range(1, 13))
plt.grid(axis='y', alpha=0.3)
plt.legend()

plt.show()

In [ ]:
# Create a summary of the Gap by Month
summary_table = monthly_climatology.copy()
summary_table['Gap'] = summary_table['Control_Avg'] - summary_table['Test']
summary_table.columns = ['Month', 'Test REP (nm)', 'Control Avg (nm)', 'Stress Gap (nm)']

print("--- FINAL MONTHLY STRESS SUMMARY (2018-2025) ---")
print(summary_table.to_string(index=False))